# Star-cluster dynamics and UV irradiation diagnostics

**Main question:** How does the dynamical state of a young stellar cluster change the UV irradiation environment of protoplanetary disks?

**Run order:** imports → initial clusters → Ph4 evolution → UV proximity → UV exposure → optional experiments.

The main simulations share the same initial star positions and masses but have different **virial ratios** ($Q=0.25,\;0.5,\;0.75$), not different initial densities. Only $Q=0.25$ and $Q=0.5$ are evolved below. Disk mass loss and VADER coupling have **not** yet been implemented. The functions `make_cluster` and `evolve_gravity` are imported from your existing `cluster_tools` module.


In [ ]:
%matplotlib inline

from amuse.lab import *
from amuse.ic.fractalcluster import new_fractal_cluster_model
from amuse.ic.kroupa import new_kroupa_mass_distribution
from amuse.community.ph4.interface import Ph4
from amuse.units import nbody_system
from cluster_tools import *
from matplotlib import pyplot as plt
import numpy as np


## 1. Build and inspect initial clusters

The clusters start with the same mass distribution and spatial configuration, with velocities scaled to the requested virial ratio. Disk hosts are defined here as stars with $M<2\,M_\odot$, and candidate strong-UV sources as stars with $M\geq8\,M_\odot$ (a simple threshold, not a full luminosity model).


In [ ]:
clusters = {
    f"Q={q}": make_cluster(r_vir=1.0 | units.pc, virial_ratio=q)
    for q in (0.25, 0.5, 0.75)
}

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, (stars, conv)) in zip(axes, clusters.items()):
    kinetic = stars.kinetic_energy().value_in(units.J)
    potential = stars.potential_energy().value_in(units.J)
    actual_q = kinetic / abs(potential)

    hosts = stars[stars.mass < (2 | units.MSun)]
    uv_sources = stars[stars.mass >= (8 | units.MSun)]
    print(
        f"{name}: N={len(stars)}, hosts(<2 Msun)={len(hosts)}, "
        f"UV sources(>=8 Msun)={len(uv_sources)}, "
        f"M_tot={stars.mass.sum().value_in(units.MSun):.0f} Msun, "
        f"R_vir={stars.virial_radius().value_in(units.pc):.2f} pc, "
        f"Q={actual_q:.2f}"
    )

    ax.scatter(stars.x.value_in(units.pc), stars.y.value_in(units.pc),
               s=2, c="grey", label="All stars")
    ax.scatter(uv_sources.x.value_in(units.pc),
               uv_sources.y.value_in(units.pc),
               s=20*np.sqrt(uv_sources.mass.value_in(units.MSun)),
               c="red", label="M >= 8 Msun")
    ax.set_title(name)
    ax.set_xlabel("x [pc]")
    ax.set_ylabel("y [pc]")
    ax.set_aspect("equal")
axes[0].legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()


## 2. Gravitational evolution with Ph4

Run the $Q=0.25$ and $Q=0.5$ clusters independently. `evolve_gravity` provides lists of `(time, Particles)` snapshots. The $Q=0.75$ initial conditions exist but are **not** evolved in this notebook.


In [ ]:
# Subvirial cluster (Q = 0.25)
stars25, conv25 = clusters["Q=0.25"]
snaps25 = evolve_gravity(stars25, conv25)

# Approximately virial cluster (Q = 0.5)
stars5, conv5 = clusters["Q=0.5"]
snaps5 = evolve_gravity(stars5, conv5)

snapshot_sets = {"Q=0.25": snaps25, "Q=0.5": snaps5}


In [ ]:
# Inspect how star positions evolve in both runs.
for label, history in snapshot_sets.items():
    fig, axes = plt.subplots(1, len(history), figsize=(4*len(history), 4))
    axes = np.atleast_1d(axes)
    for ax, (t, snapshot) in zip(axes, history):
        uv_sources = snapshot[snapshot.mass >= (8 | units.MSun)]
        ax.scatter(snapshot.x.value_in(units.pc),
                   snapshot.y.value_in(units.pc), s=1, c="grey")
        ax.scatter(uv_sources.x.value_in(units.pc),
                   uv_sources.y.value_in(units.pc), s=10, c="red")
        ax.set_title(f"{label}, t={t.value_in(units.Myr):.2f} Myr")
        ax.set_xlim(-5, 5)
        ax.set_ylim(-5, 5)
        ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()


## 3. UV distance and exposure helper functions

Distances are calculated in **3D**. The exposure proxy $I_i=\sum_j (1\,\mathrm{pc}/d_{ij})^2$ counts all massive sources with equal weight. It is **not** a physical UV flux or disk mass-loss rate. A small floor prevents division by zero in coincident positions.


In [ ]:
# Define diagnostics explicitly so they do not depend on undocumented
# helper functions inside cluster_tools.
def _xyz_pc(particles):
    return np.column_stack((
        particles.x.value_in(units.pc),
        particles.y.value_in(units.pc),
        particles.z.value_in(units.pc),
    ))


def _disk_host_source_distance_squared(stars, disk_mass_limit=2,
                                       source_mass_limit=8):
    hosts = stars[stars.mass < (disk_mass_limit | units.MSun)]
    sources = stars[stars.mass >= (source_mass_limit | units.MSun)]
    if len(sources) == 0:
        raise ValueError("No stars above the UV-source mass threshold")
    if len(hosts) == 0:
        raise ValueError("No stars below the disk-host mass threshold")
    delta = _xyz_pc(hosts)[:, None, :] - _xyz_pc(sources)[None, :, :]
    return np.sum(delta**2, axis=2)  # pc^2, numerical values


def nearest_uv_distance(stars):
    return np.sqrt(_disk_host_source_distance_squared(stars).min(axis=1))


def exposure_proxy(stars):
    distance_sq_pc2 = _disk_host_source_distance_squared(stars)
    return np.sum(1.0 / np.maximum(distance_sq_pc2, 1e-12), axis=1)


## 4. UV-source proximity: initial conditions and evolution

**Left:** Cumulative distribution of distances from disk hosts to their nearest UV source. **Right:** Fraction of hosts within 0.1 pc over time, compared across evolved virial ratios. The 0.1 pc cutoff is only a diagnostic threshold, **not** a photoevaporation boundary.

Because initial positions are the same, expect nearly identical initial-distance curves for all three $Q$ values; the distinction should emerge during dynamical evolution.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Initial stellar positions: same for all Q, as expected.
for name, (stars, _) in clusters.items():
    distances = nearest_uv_distance(stars)
    sorted_d = np.sort(distances)
    cumulative = np.arange(1, len(sorted_d)+1) / len(sorted_d)
    ax[0].plot(sorted_d, cumulative, label=name)
    print(f"{name}: median nearest-source distance = {np.median(distances):.3f} pc; "
          f"within 0.1 pc = {np.mean(distances < 0.1):.2%}")

ax[0].set_xscale("log")
ax[0].set_xlabel("Distance to nearest UV source [pc]")
ax[0].set_ylabel("Cumulative fraction of disk hosts")
ax[0].set_title("Initial UV-source proximity")
ax[0].legend()
ax[0].grid(alpha=0.3)

# Time evolution: iterate over *each* list of (time, snapshot) pairs.
for label, history in snapshot_sets.items():
    t_myr = []
    close_fraction = []
    for t, snapshot in history:
        distances = nearest_uv_distance(snapshot)
        t_myr.append(t.value_in(units.Myr))
        close_fraction.append(np.mean(distances < 0.1))
    ax[1].plot(t_myr, close_fraction, "o-", label=label)

ax[1].set_xlabel("Time [Myr]")
ax[1].set_ylabel("Fraction within 0.1 pc")
ax[1].set_title("UV-source proximity versus virial ratio")
ax[1].set_ylim(0, 1)
ax[1].legend()
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Geometric UV exposure over time

Evaluate the **median** dimensionless exposure proxy among disk hosts at every saved snapshot. Compare the two available Ph4 runs, not a nonexistent `snaps` variable. This is the input geometry for a future model that assigns UV luminosities and physical photoevaporation rates.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for label, history in snapshot_sets.items():
    t_myr = []
    median_exposure = []
    for t, snapshot in history:
        values = exposure_proxy(snapshot)
        t_myr.append(t.value_in(units.Myr))
        median_exposure.append(np.median(values))
    ax.plot(t_myr, median_exposure, "o-", label=label)
    print(f"{label}: median exposure at final snapshot = {median_exposure[-1]:.3g}")

ax.set_yscale("log")
ax.set_xlabel("Time [Myr]")
ax.set_ylabel("Median geometric exposure proxy [dimensionless]")
ax.set_title("Irradiation environment versus virial ratio")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Next modelling step

The current results describe *where radiation sources are* relative to potential disk hosts, **not** how much disk mass is removed. A future coupled model needs stellar UV luminosities (possibly informed by SeBa), a validated irradiation-to-mass-loss prescription, and disk evolution (e.g., VADER). The cells below are exploratory drafts, not part of the validated main workflow.

---

## Appendix A — stand-alone fractal-cluster initialization experiment

This separate experiment explores AMUSE's `new_fractal_cluster_model` and is not used by the `make_cluster` / Ph4 analysis above. The original sequence is preserved here at the end so it does not interrupt the main workflow.


In [ ]:
# Independent exploratory model; not used in the main Ph4 calculations above.
n_stars = 1000
mass_min = 0.01 | units.MSun
mass_max = 100 | units.MSun
masses_kroupa = new_kroupa_mass_distribution(
    n_stars, mass_min=mass_min, mass_max=mass_max
)


In [ ]:
fractal_dimension = 2.0
length_scale = 1.0 | units.parsec
mass_scale = masses_kroupa.sum()

# This converter sets the N-body mass and length scales in physical units.
fractal_converter = nbody_system.nbody_to_si(mass_scale, length_scale)
star_cluster = new_fractal_cluster_model(
    n_stars, fractal_dimension=fractal_dimension, convert_nbody=fractal_converter
)

star_cluster.mass = masses_kroupa
star_cluster.radius = 0.5 | units.au
star_cluster.move_to_center()
print(star_cluster[:5])


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(
    star_cluster.x.value_in(units.pc),
    star_cluster.y.value_in(units.pc),
    c=np.log10(star_cluster.mass.value_in(units.MSun)),
    s=25.0*np.sqrt(star_cluster.mass.value_in(units.MSun)),
)
plt.xlabel("x [pc]")
plt.ylabel("y [pc]")
plt.title("Standalone fractal cluster test")
plt.axis("equal")
plt.show()


## Appendix B — experimental protoplanetary-disk code (not yet coupled)

These blocks were in the source notebook as exploratory cells. They are kept **as reference snippets, not executable cells**, because they are not part of the current Ph4/UV analysis. In particular, the sample code refers to an undefined `particles_fractal`, creates disks without validated host assignment, or starts VADER without a configured model. Making them runnable would be a separate scientific development task.


### Disk-helper draft

This original draft is preserved verbatim; it was commented out in the supplied notebook.

```python
from amuse.datamodel import Particles
from amuse.ext.protodisk import ProtoPlanetaryDisk
'''

def make_disk(star, n_disk=1000, r_in=1.0, r_out=100.0, disk_frac=0.01): 
    """Return a gas disk centred on `star`. r_in and r_out are in AU.""" 
    converter = nbody_system.nbody_to_si(star.mass, r_in | units.AU) 
    disk = ProtoPlanetaryDisk(n_disk, convert_nbody=converter, Rmin=1.0, Rmax=r_out / r_in,discfraction=disk_frac).result 

    disk.position += star.position # move disk onto the star 
    disk.velocity += star.velocity 
    disk.host_key = star.key # tag disk particles with their star 
    return disk 

disks = Particles() 
for star in particles_fractal: 
    if star.mass < 2 | units.MSun: 
        disks.add_particles(make_disk(star))
    else: 
        None
'''
```


### Single-disk example and population loop

This original experimental snippet is preserved verbatim. It has not been tested as a population-level disk implementation.

```python
from amuse.community.seba import Seba
from amuse.ext.protodisk import ProtoPlanetaryDisk
import numpy as np

# this coding snippet will devlop a proto planetary disk for one star which 
# we need to adapt to use it for all 1000 stars! 

disk = ProtoPlanetaryDisk(targetN=100, densitypower=1.5,radius_min=0.1, radius_max=10, Rmin=0.1, Rmax=10, q_out=1, discfraction=0.1,).result
disk.rotate(0, np.pi/6, np.pi/6)

for name in big:
    disk = ProtoPlanetaryDisk(targetN=1, densitypower=1, radius_min=0.1, radius_max=10, Rmin=0.1, Rmax=10, q_out=1, discfraction=0.1,).result
```


### VADER initialization test

Starting `Vader()` alone is not yet disk evolution. This snippet is preserved but not executed on **Run All**.

```python
from amuse.community.vader.interface import Vader

Vader()
```
